---
---
# Forecasting Model

Objectives:- 
- Sales Forecasting

---
Import Libraries:-

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import numpy as np
from scipy.interpolate import make_interp_spline
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler

---
Load Clean Data:-

In [ ]:
file_path = ('../Data/02_Clean_Data/MM_Clean_Data.csv')
df = pd.read_csv(file_path)

df["Date"] = pd.to_datetime(df["Date"])
df.head()

---
Revenue Forecast:-

In [ ]:
df = df.sort_values("Date")
monthly = df.resample("ME", on = "Date")["Revenue_USD"].sum()

model_fit = ARIMA(monthly, order = (1, 1, 1)).fit()
forecast = model_fit.get_forecast(steps = 5)

forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

fig, ax = plt.subplots(figsize = (14, 5))

ax.plot(monthly.index, monthly.values, marker = "o", label = "Actual")

ax.plot(forecast_mean.index, forecast_mean.values,
        marker = "o", linestyle = "--", label = "Forecast")

ax.fill_between(conf_int.index,
                conf_int.iloc[:, 0],
                conf_int.iloc[:, 1],
                alpha = 0.2,
                label = "95% Confidence Interval")

ax.axvline(forecast_mean.index[0], linestyle = ":")
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x/1e6:.0f}M"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval = 2))


ax.set_title("Monthly Revenue Forecast — ARIMA(1,1,1)")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (USD)")
ax.legend(loc = "lower center", bbox_to_anchor = (0.5, -0.3), ncol = 6)

plt.xticks(rotation = 0)
plt.tight_layout()
plt.show()

---
EV Demand Forecast:-

In [ ]:
df = df.sort_values("Date")
EV_Data = df[df["Vehicle_Type"] == "EV"]
EV_Trend = EV_Data.resample("ME", on="Date")["Units_Sold"].sum()

scaler = StandardScaler()
EV_scaled = scaler.fit_transform(EV_Trend.values.reshape(-1, 1)).flatten()

model = ARIMA(EV_scaled, order = (2, 1, 2))

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_fit = model.fit(method = "statespace") 


print(f"Converged : {model_fit.mle_retvals.get('converged', 'N/A')}")
print(f"AIC       : {model_fit.aic:.2f}")

forecast = model_fit.get_forecast(steps = 6)
forecast_mean_scaled = forecast.predicted_mean
conf_int_scaled = forecast.conf_int()

forecast_mean = scaler.inverse_transform(forecast_mean_scaled.reshape(-1, 1)).flatten()

conf_lower = scaler.inverse_transform(conf_int_scaled[:, 0].reshape(-1, 1)).flatten()

conf_upper = scaler.inverse_transform(conf_int_scaled[:, 1].reshape(-1, 1)).flatten()

last_date = EV_Trend.index[-1]
forecast_index = pd.date_range(last_date, 
                    periods = 7, 
                    freq = "ME")[1:]

x_num = np.arange(len(forecast_index))
x_smooth = np.linspace(0, len(forecast_index) - 1, 200)
spline = make_interp_spline(x_num, forecast_mean, k = 3)
forecast_smooth = spline(x_smooth)
date_smooth = pd.date_range(forecast_index[0],
                            forecast_index[-1], 
                            periods = 200)

fig, ax = plt.subplots(figsize = (14, 5))

ax.plot(EV_Trend.index, 
        EV_Trend.values,
        marker = "o",
        linewidth = 2,
        markersize = 4, 
        color = "steelblue",
        label = "Actual EV Sales")

ax.plot(date_smooth,
        forecast_smooth,
        linestyle = "--",
        linewidth = 2,
        color="orange",
        label = "Forecast")

ax.fill_between(forecast_index,
                conf_lower,
                conf_upper,
                alpha = 0.15, 
                color = "steelblue", 
                label = "95% Confidence Interval")

ax.axvline(forecast_index[0],
           linestyle = ":",
           color = "gray", 
           linewidth = 1.2)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval = 2))

ax.set_title("Monthly EV Demand Forecast — ARIMA(2,1,2)", fontsize = 13, pad = 12)
ax.set_xlabel("Month")
ax.set_ylabel("Units Sold")
ax.legend(loc = "lower center", bbox_to_anchor = (0.5, -0.3), ncol = 3)

plt.xticks(rotation = 0)
plt.tight_layout()
plt.show()

---
---